# 旧 MXIM IBKR 历史价格来源预检
仅一条冻结只读请求，精确保留连接器响应；不读账户或 SOXX 结果。

In [1]:
import hashlib,json,shlex,subprocess,tarfile,tempfile
from pathlib import Path
from src.data.nas_market_data import _settings,nas_api_call,_ssh
root=Path.cwd().resolve();card=root/'research/stage1_mxim_ibkr_delisted_price_source_probe_20261023';plan=json.loads((card/'plan.json').read_text());raw=root/'data/snapshots/stage1_mxim_ibkr_delisted_price_source_probe_20261023/ibkr_response.json';response=json.loads(raw.read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and plan['request']['max_calls']==1 and response['structuredContent']['error']=='Details currently unavailable. Please try again later and contact client services if the issue persists.'
assert response['content']==[{'type':'text','text':json.dumps(response['structuredContent'],separators=(',',':'))}]
settings=_settings(root);paths=[card/'plan.json',raw]
with tempfile.TemporaryDirectory(prefix='mxim-ibkr-source-') as directory:
 archive=Path(directory)/'response.tar.gz'
 with tarfile.open(archive,'w:gz') as output:
  for p in paths:output.add(p,arcname=str(p.relative_to(root)),recursive=False)
 digest=sha(archive);evidence=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
 assert evidence['sha256']==digest and evidence['size_bytes']==archive.stat().st_size
 remote_path=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(remote_path)).split()[0]==digest
 restored=Path(directory)/'restored.tar.gz'
 with restored.open('wb') as out:transfer=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote_path)],stdout=out,stderr=subprocess.PIPE,timeout=60)
 assert transfer.returncode==0 and sha(restored)==digest
 with tarfile.open(restored,'r:gz') as source:
  members={m.name:m for m in source.getmembers()};assert set(members)=={str(p.relative_to(root)) for p in paths}
  for p in paths:assert source.extractfile(members[str(p.relative_to(root))]).read()==p.read_bytes()
 evidence['nas_path']=remote_path
result={'status':'needs_market_data_IBKR_old_MXIM_history_details_unavailable','plan_sha256':sha(card/'plan.json'),'connector_search_result':plan['prior_discovery']['exact_result'],'history_request':plan['request'],'history_calls':1,'returned_bars':0,'connector_error':response['structuredContent']['error'],'connector_isError_flag':response['isError'],'raw_response_relative_path':str(raw.relative_to(root)),'raw_response_sha256':sha(raw),'NAS_evidence':evidence,'NAS_scoped_restore_verified':True,'market_outcome_reads':0,'account_reads':0,'order_calls':0,'model_fits':0,'native_accounts':0,'historical_2017_price_availability_concluded':False}
(card/'source_result.json').write_text(json.dumps(result,indent=2)+'\n')
summary={'experiment':plan['experiment'],'decision':result['status'],'objective_achieved':False,'plan_sha256':result['plan_sha256'],'source_result':'source_result.json','history_calls':1,'returned_bars':0,'NAS_archive_sha256':evidence['sha256'],'NAS_scoped_restore_verified':True,'model_fits':0,'native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':[]}}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
print(summary['decision'],evidence['sha256'])

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


needs_market_data_IBKR_old_MXIM_history_details_unavailable 1c4a26d6456bc13acdffdd4abcf83ca6b387378fa6cf2ef54eb3a43a1e0fa925


## 中文结论
# 旧 Maxim（MXIM）IBKR 历史价格来源：接口未返回明细

只读 `search_contracts("MXIM")` 返回 `MAXIM INTEGRATED PRODUCTS` 股票旧合约 ID `272207`；这个发现调用发生在冻结历史请求计划前，原始搜索响应未留存，故它只定位本次单合约探针，不构成已认证的旧证券映射。随后按冻结计划向该 ID 请求一次最多 1000 根日线、包含公司行动的历史图表；连接器仅返回 `Details currently unavailable. Please try again later and contact client services if the issue persists.`，没有任何价格 bar。没有重试、账户读取、订单、拟合或账户。

完整历史请求返回 JSON 的原字节和 SHA-256 已保存在本机并归档 Yanxuan NAS，远端哈希与限定恢复通过。决定：`needs_market_data_IBKR_old_MXIM_history_details_unavailable`。这是连接器的访问/服务失败，**不能证明旧 MXIM 的 2017 年价格不存在**，也不能修复 2017 原 SOXX 篮子至少 14.2745% 的现有 Qlib 成分价格身份缺口。后续需要另行冻结并验证其他历史证券数据来源；不按本次错误提示立即重试。SOXX v2 目标仍未达成。
